# [SOLUTION] Mini-Workshop 2 — Decision Tree & the Honest Test

**Dataset:** Melbourne housing · **Time:** ~30 min · **Level:** beginner

The cleaning is done for you (same as Mini-Workshop 1). You focus on **one new model** and
one big idea: **cross-validation** — why a model that looks perfect on training data can still be bad.

### Rules
- Run the provided setup cell first (do not edit it).
- Fill in every `# TODO`. Keep `random_state=42`.

## Setup (provided — just run)

In [ ]:
# ============================================================
# PROVIDED SETUP  —  just run this cell, do not edit.
# It repeats the Explore + Clean steps you did in Mini-Workshop 1,
# using the SAME pipeline-free style from the slides, and hands you:
#   X_train_prepared, X_test_prepared, y_train, y_test, feature_names
# ============================================================
import os
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error

URL = "https://raw.githubusercontent.com/NUS-ISS-SS/mla-day1-workshop-student/development/datasets/housing/melb_data.csv"
LOCAL = "./datasets/housing/melb_data.csv"
df = pd.read_csv(LOCAL) if os.path.exists(LOCAL) else pd.read_csv(URL)

TARGET = "Price"
NUM_COLS = ["Rooms","Distance","Postcode","Bedroom2","Bathroom","Car",
            "Landsize","BuildingArea","YearBuilt","Lattitude","Longtitude","Propertycount"]
CAT_COLS = ["Type","Method","Regionname"]   # low-cardinality text only

y = df[TARGET]
X = df[NUM_COLS + CAT_COLS]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# fit cleaners on TRAIN only, then reuse them on test (no data snooping)
num_imputer = SimpleImputer(strategy="median").fit(X_train[NUM_COLS])
scaler      = StandardScaler().fit(num_imputer.transform(X_train[NUM_COLS]))
cat_imputer = SimpleImputer(strategy="most_frequent").fit(X_train[CAT_COLS])
encoder     = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(cat_imputer.transform(X_train[CAT_COLS]))

def prepare(frame):
    num = scaler.transform(num_imputer.transform(frame[NUM_COLS]))
    cat = encoder.transform(cat_imputer.transform(frame[CAT_COLS]))
    return np.hstack([num, cat])

X_train_prepared = prepare(X_train)
X_test_prepared  = prepare(X_test)
feature_names = NUM_COLS + list(encoder.get_feature_names_out(CAT_COLS))

def cv_rmse(model):
    scores = cross_val_score(model, X_train_prepared, y_train,
                             scoring="neg_mean_squared_error", cv=10)
    return np.sqrt(-scores)

print("Setup done. X_train_prepared:", X_train_prepared.shape)

## 1) Train a Decision Tree and look at its **training** error
Train on `X_train_prepared`, then compute RMSE on that same training data.

In [ ]:
# TODO: train a Decision Tree (random_state=42) on the prepared training data
tree_reg = DecisionTreeRegressor(random_state=42).fit(X_train_prepared, y_train)  # <<

# TODO: predictions on the TRAINING data, then training RMSE
train_pred = tree_reg.predict(X_train_prepared)  # <<
tree_train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))  # <<

print("Decision Tree TRAINING RMSE: $", round(tree_train_rmse))
# Notice how tiny this is. Suspicious?

## 2) The honest test — cross-validation
A near-zero training error usually means the model **memorised** the data. Cross-validation checks how it does on data it *hasn't* seen. Use the provided `cv_rmse(...)` helper (10-fold).

In [ ]:
# TODO: cross-validated RMSE scores for a fresh Decision Tree
tree_cv = cv_rmse(DecisionTreeRegressor(random_state=42))  # <<

assert len(tree_cv) == 10
print("Decision Tree CV RMSE scores:", np.round(tree_cv).astype(int))
print("Mean CV RMSE: $", round(tree_cv.mean()))

## 3) Reflect
Compare the two numbers you just got:

- **Training RMSE** was near zero.
- **Cross-validated RMSE** is much larger.

That gap is **overfitting**: the tree memorised the training rows instead of learning general patterns. The CV number is the honest one. In Mini-Workshop 3 you'll meet a model designed to shrink this gap.